# Python companion — *Lean for Working Algebraists*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abderrahim-lectures/lean4-learning/blob/master/lean_book/python-companion/python_companion.ipynb)

This notebook collects every "Programmer's corner (Python)" snippet from the book, in reading order, as runnable and editable code. It does not require a Lean installation. Each section names the book chapter it comes from and states the point the snippet makes; the full explanation is in the book itself.

This is a companion to the Lean material, not a substitute for it — the book's argument is about what Lean's type system checks statically that Python's does not, and reading these examples run is meant to make that argument concrete, not to replace it.

## Chapter 1 §1 — static type checking versus a runtime check

`add_them` runs without complaint on a boolean argument, silently treating `True` as `1`, and only fails once the string case actually executes. Nothing about the function's text rules either call out in advance.

In [ ]:
# Python: this line is perfectly legal to *write*.
def add_them(a, b):
    return a + b

print(add_them(3, True))    # 4 -- Python silently treats True as 1, no error at all
print(add_them(3, "oops"))  # TypeError: unsupported operand type(s), but only once this exact line runs

## Chapter 1 §4 — a function that silently returns a wrong answer

`dot` computes a dot product using `zip`, which truncates to the shorter of its two arguments instead of raising an error. The function's signature states nothing about the two lists needing equal length, so this mistake is never caught by the type system, only by a human noticing the number looks wrong. This is the motivating example for Lean's dependent types (`Vec`) later in the chapter.

In [ ]:
def dot(xs, ys):
    return sum(x * y for x, y in zip(xs, ys))

print(dot([1, 2, 3], [4, 5, 6]))   # 32 -- correct
print(dot([1, 2, 3], [4, 5]))      # 14 -- silently wrong: zip() truncates to the shorter list

## Chapter 1 §5 — what Python's `TypeVar` cannot express

`TypeVar` makes a function generic over a *type*. It cannot make a function generic over a *value* that also appears in the return type, which is exactly what `Vec.replicate`'s Lean signature requires (the length `n` is a value, not a type, and it appears in both the argument and the result type).

In [ ]:
from typing import TypeVar
T = TypeVar("T")

def identity(x: T) -> T:      # generic over a TYPE -- fine, this is TypeVar's job
    return x

print(identity(5))
print(identity("hello"))

# There is no Python type-hint equivalent of:
#   def replicate(a: T, n: <this specific Nat>) -> Vec[T, <that same Nat>]
# because "the type mentions n's VALUE" is not expressible by any
# combination of ordinary hints or TypeVar.

## Chapter 4 §5 — structural recursion, matching `induction`

A hand-rolled Peano-style natural number (`Zero`/`Succ`) and a recursive `add` function over it have exactly the same two-case shape as Lean's `induction b with | zero => ... | succ k ih => ...`: a base case, and a recursive call that plays the role of the induction hypothesis `ih`.

In [ ]:
class Zero: pass
class Succ:
    def __init__(self, pred): self.pred = pred

def add(a, b):
    if isinstance(b, Zero):
        return a                           # base case, like `| zero =>`
    return Succ(add(a, b.pred))            # recursive call, like `| succ k ih =>`

def to_int(n):
    count = 0
    while isinstance(n, Succ):
        count += 1
        n = n.pred
    return count

two = Succ(Succ(Zero()))
three = Succ(Succ(Succ(Zero())))
print(to_int(add(two, three)))  # 5

## Chapter 5 §3 (a) — `mypy` as a light version of a typing judgment

`mypy` checks `to_str`'s argument type against its declared signature before the program runs, rejecting a call with a `str` argument without executing anything. This is the same kind of check as Lean's typing rules, applied to a language most readers already know.

In [ ]:
def apply_twice(f: int, x: int) -> int:  # pretend f is Callable[[int], int]
    ...

def to_str(n: int) -> str:
    return str(n)

# mypy accepts this: to_str's declared input type (int) matches what
# it is given (an int literal).
print(to_str(5))

# mypy rejects this with an error, WITHOUT running anything -- but Python
# itself will still run it and raise at runtime instead, since Python does
# not enforce type hints on its own:
# error: Argument 1 to "to_str" has incompatible type "str"; expected "int"
print(to_str("already a string"))

## Chapter 5 §3 (b) — `TypeVar` as an escape hatch from monomorphism

Without `TypeVar`, a function typed against one specific type cannot be reused at another type without writing a separate copy for each type. `TypeVar` lets one signature stay valid at every type — the same gap that motivates Lean's `{α : Type}` implicit argument.

In [ ]:
from typing import TypeVar
T = TypeVar("T")

def identity(x: T) -> T:   # one signature, valid at every type
    return x

print(identity(5))        # T := int
print(identity("hello"))  # T := str

## Chapter 5 §3 (c) — why `Type : Type` is not a Python-style question

In Python, `type(type)` is `type` itself: there is no stratification, because Python's type system is not used as a proof system, and nothing checks proofs against it. Lean's `Type` cannot self-apply this way — `Type : Type` is inconsistent, since it permits encoding a version of Girard's paradox and proving `False`.

In [ ]:
print(type(3))
print(type(int))
print(type(type))

## Chapter 8 §5 — brute-force enumeration, matching `by decide`

Checking a finite ring's axioms (e.g. associativity on `Fin 3`) by `by decide` in Lean brute-force-enumerates every combination of the finitely many elements and checks the equation on each — exactly what the loop below does. The difference is not the algorithm, but where the check lives: this Python assertion only runs if some line of code calls it, while Lean's `decide` runs as part of type-checking the definition itself, so a value with a failing check cannot exist at all.

In [ ]:
domain = range(3)
print(all((a + b) + c == a + (b + c) for a in domain for b in domain for c in domain))

## Where to go next

These snippets are illustrations of a point the book makes about Lean's type system, not a substitute for the Lean material itself. To follow the book's own examples in Lean, see [`lean_project/`](../../lean_project/) for a companion Lean 4 project containing every code block from the book, verified against the pinned toolchain with `lake build`.